In [4]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Подключаем корень проекта
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.llm_platform.data_foundry.dataset_generator import DatasetGenerator
import src.llm_platform.config as config

print("Ключ API загружен:", bool(config.OPENROUTER_API_KEY))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Ключ API загружен: True


In [5]:
# Настраиваем пути
raw_dir = project_root / "data" / "raw"
output_dir = project_root / "data" / "processed"

# Системный промпт для генерации SFT-пар
system_prompt = (
    "You are an expert AI data generator. "
    "Your task is to read the provided text chunk and extract one highly specific, "
    "complex question and its detailed answer. Ensure the answer is fully contained "
    "within the source text."
)

# Инициализируем генератор (максимум 3 одновременных запроса для бесплатного API)
generator = DatasetGenerator(
    raw_data_dir=raw_dir,
    output_dir=output_dir,
    max_concurrent_requests=3
)

# Запускаем генерацию
print("Запуск пайплайна...")
await generator.generate_dataset(
    system_prompt=system_prompt, 
    output_filename="test_sft_dataset.jsonl"
)

2026-05-31 22:53:16 [INFO] (src.llm_platform.data_foundry.llm_client): LLMClient initialized with model: openai/gpt-oss-120b


2026-05-31 22:53:16 [INFO] (src.llm_platform.data_foundry.dataset_generator): Extracting chunks from: moesd.pdf


Запуск пайплайна...


2026-05-31 22:53:16 [INFO] (src.llm_platform.data_foundry.dataset_generator): Total chunks extracted: 111. Starting LLM generation...
2026-05-31 22:53:17 [INFO] (httpx): HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-31 22:53:17 [INFO] (httpx): HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-31 22:53:17 [INFO] (httpx): HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-31 22:53:18 [INFO] (httpx): HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-31 22:53:19 [INFO] (httpx): HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-31 22:53:29 [INFO] (httpx): HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-31 22:53:41 [INFO] (httpx): HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-31 22:53:43 [INFO] (httpx

In [ ]:
import json

output_file = output_dir / "test_sft_dataset.jsonl"

if output_file.exists():
    print(f"Файл успешно создан! Размер: {output_file.stat().st_size} байт\n")
    print("Первые 2 сгенерированные пары:\n")
    
    with open(output_file, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 2:
                break
            data = json.loads(line)
            print(json.dumps(data, indent=2, ensure_ascii=False))
            print("-" * 40)
else:
    print("Файл не найден. Что-то пошло не так на этапе генерации.")